In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Kiểm tra GPU, đồng bộ code mới nhất vào /content/EEG-mini-project và cài xgboost
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_GIT_URL = "https://github.com/Longkyoladn/EEG-mini-project.git"
REPO_BRANCH = "main"
COLAB_REPO_DIR = Path("/content/EEG-mini-project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/EEG-mini-project")
LOCAL_FEATURES_DIR = Path("/content/features")

subprocess.run(["nvidia-smi"], check=False)

if not COLAB_REPO_DIR.exists():
    if (DRIVE_PROJECT_DIR / "src").is_dir() and (DRIVE_PROJECT_DIR / "splits").is_dir():
        print(f"Copying repo code from Drive {DRIVE_PROJECT_DIR} -> {COLAB_REPO_DIR} ...")
        shutil.copytree(
            DRIVE_PROJECT_DIR,
            COLAB_REPO_DIR,
            ignore=shutil.ignore_patterns("features", "results", "data", "oddball_data", ".git"),
        )
    else:
        subprocess.run(
            ["git", "clone", "-b", REPO_BRANCH, REPO_GIT_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
elif (COLAB_REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull"], check=False)

os.chdir(COLAB_REPO_DIR)
if str(COLAB_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(COLAB_REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0.0"], check=True)
print(f"Ready at: {Path.cwd()}")

Ready at: /content/EEG-mini-project


In [3]:
# 2. Kiểm tra Google Drive & copy features/temporal + features/wavelet sang /content/features
# (Các file đã copy sẵn trong /content/features sẽ được giữ nguyên, không copy lại)
from scripts.training.ML3.train_ml3 import main as train_ml3_main, setup_colab_drive_paths

paths = setup_colab_drive_paths(
    drive_project_dir=DRIVE_PROJECT_DIR,
    stage_to_content=True,
    local_features_dir=LOCAL_FEATURES_DIR,
)
for k, v in paths.items():
    print(f"{k}: {v}")

[Stage Features] /content/drive/MyDrive/EEG-mini-project/features -> /content/features | copied=78, reused=0, transferred=962.5 MB
project_dir: /content/drive/MyDrive/EEG-mini-project
drive_features_dir: /content/drive/MyDrive/EEG-mini-project/features
features_dir: /content/features
temporal_dir: /content/features/temporal
wavelet_dir: /content/features/wavelet
output_dir: /content/drive/MyDrive/EEG-mini-project/results/ml3


In [ ]:
# 3. Kiểm tra nhanh cấu hình & số lượng feature (--plan-only)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "cs5",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--plan-only",
    ]
)

In [ ]:
# 4. Chạy Cross-Subject 5-Fold (5 kịch bản chính)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "cs5",
        "--experiments", "temporal", "wavelet_w5", "wavelet_w8", "combined_w5", "combined_w8",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)

In [ ]:
# 5. Chạy LOSO (3 kịch bản chính: temporal, combined_w5, combined_w8)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "loso",
        "--experiments", "temporal", "combined_w5", "combined_w8",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)

In [4]:
# 6. Chạy Within-Subject 5-Fold (chỉ chạy kịch bản tốt nhất: combined_w5)
# 39 subjects x 5 folds = 195 folds; tự động lưu checkpoint, metrics, oof_predictions.npz và 6 ảnh 300 DPI lên Drive
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "within",
        "--experiments", "combined_w5",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)

[ML3][WITHIN][combined_w5] samples=20,886, features=10,287, folds=195, device=cuda


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [07:57:54] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  sub-01_fold_00 [trained] best_iter=174 thr=0.352 PR-AUC=0.5957 ROC-AUC=0.8715 BalAcc=0.6927 F1=0.5000 time=3.1s
  sub-01_fold_01 [trained] best_iter=16 thr=0.385 PR-AUC=0.7673 ROC-AUC=0.9002 BalAcc=0.7812 F1=0.4167 time=1.7s
  sub-01_fold_02 [trained] best_iter=248 thr=0.203 PR-AUC=0.7535 ROC-AUC=0.9332 BalAcc=0.8333 F1=0.6207 time=2.8s
  sub-01_fold_03 [trained] best_iter=7 thr=0.435 PR-AUC=0.5290 ROC-AUC=0.7883 BalAcc=0.7425 F1=0.4000 time=1.6s
  sub-01_fold_04 [trained] best_iter=238 thr=0.131 PR-AUC=0.7141 ROC-AUC=0.9228 BalAcc=0.8741 F1=0.5641 time=3.3s
  sub-03_fold_00 [trained] best_iter=20 thr=0.460 PR-AUC=0.9449 ROC-AUC=0.9863 BalAcc=0.9271 F1=0.7692 time=1.8s
  sub-03_fold_01 [trained] best_iter=3 thr=0.525 PR-AUC=0.7126 ROC-AUC=0.9160 BalAcc=0.7031 F1=0.5385 time=1.7s
  sub-03_fold_02 [trained] best_iter=246 thr=0.292 PR-AUC=0.9655 ROC-AUC=0.9932 BalAcc=0.9359 F1=0.9091 time=2.4s
  sub-03_fold_03 [trained] best_iter=142 thr=0.250 PR-AUC=0.9492 ROC-AUC=0.9678 BalAcc=0.9530 

0

In [ ]:
# 7. (Tùy chọn) Hiển thị nhanh toàn bộ ảnh 300 DPI đã tạo của within / cs5 / loso
from IPython.display import Image, display

PROTOCOLS_TO_SHOW = ("within",)  # Có thể đổi thành ("cs5", "loso", "within")
for proto in PROTOCOLS_TO_SHOW:
    p_dir = paths["output_dir"] / proto
    pngs = sorted(p_dir.glob("*.png"))
    print(f"=== {proto.upper()}: {len(pngs)} figure(s) in {p_dir} ===")
    for img_path in pngs:
        print(f"  -> {img_path.name}")
        display(Image(filename=str(img_path)))